# ATHOS Iris API User Guide Generation

Let's use ATHOS and a draft Iris MCP tool to generate a user guide for the Iris API, which we can then use to refine our MCP functionality.

## Imports

In [1]:
import os
import sys

from dotenv import load_dotenv

from athos import build_config, build_flow
from athos.notebook_utils import add_conda_bin_to_path

## Environment setup

In [2]:
# Load some settings from a .env file
load_dotenv('../.env')
# Setup path for launching MCP servers
add_conda_bin_to_path()

# Setup CBorg
api_base="https://api.cborg.lbl.gov"
api_key_path = '~/.cborg/api_key'

with open(os.path.expanduser(api_key_path)) as f:
    api_key = f.read().strip()

os.environ['CBORG_API_URL'] = api_base
os.environ['CBORG_API_KEY'] = api_key

## Configure agents

In [3]:
from athos.main import _format_config

# Load ATHOS configuration
config = build_config('../config/config.yaml')
print(_format_config(config))

Configuration Summary:
Models:
  - gpt_4o: openai/gpt-4o
  - gpt_5: openai/gpt-5
  - cborg_chat: lbl/cborg-chat:latest
  - cborg_coder: lbl/cborg-coder:latest
  - claude_sonnet: anthropic/claude-sonnet
  - o4_mini: openai/o4-mini
Tools:
  - websearch: smolagents.default_tools.DuckDuckGoSearchTool
  - webvisit: smolagents.default_tools.VisitWebpageTool
MCP Servers:
  - filesystem_mcp: mcp_stdio
  - servicenow_mcp: mcp_stdio
  - sfapi_mcp: mcp_stdio
  - iris_mcp: mcp_stdio
  - docs_rag_mcp: mcp_stdio
  - consulting_docs_rag_mcp: mcp_stdio
Agent Flows:
  - hpc_chatbot: chatbot
  - web_agent: agent
  - filesystem_agent: agent
  - servicenow_agent: agent
  - sfapi_agent: agent
  - docs_rag_agent: agent
  - iris_agent: agent
  - hpc_consultant: agent


In [4]:
# Select agent flow to deploy
flow_name = 'iris_agent'

In [5]:
# Make some on-the-fly adjustments to our flow config
flow_config = config.execution_flows[flow_name]

flow_config.max_steps = 20
flow_config.additional_authorized_imports += ['json']

# Add filesystem tools
flow_config.mcp_servers.append('filesystem_mcp')

In [6]:
# Construct the agent flow
flow = build_flow(flow_name, config)

2025-08-27 14:34:49,379 - athos.factories - INFO - Created MCP client 'iris_mcp'
2025-08-27 14:34:50,461 - athos.factories - INFO - Created MCP client 'filesystem_mcp'


In [7]:
flow.agent.visualize()

CodeAgent | anthropic/claude-sonnet
├── ✅ Authorized imports: ['json']
└── 🛠️ Tools:
    ┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ Name                     ┃ Description                             ┃ Arguments                              ┃
    ┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
    │ get_user_info                │ Retrieve users from the Iris system.    │ uid (`string`): see tool description   │
    │                          │                                         │ year (`string`): see tool description  │
    │                          │ Args:                                   │ username (`string`): see tool          │
    │                          │     uid: Optional user ID to filter by  │ description                            │
    │                          │     year: Optional year to query        │                                        │
    │                          │ historical data                         │                                        │
    │                          │     username: Optional user name to     │                                        │
    │                          │ filter by                               │                                        │
    │                          │                                         │                                        │
    │                          │ Returns:                                │                                        │
    │                          │     JSON string containing user data    │                                        │
    │ get_user_allocations     │ Get users with their compute            │ uid (`string`): see tool description   │
    │                          │ allocations per project.                │ year (`string`): see tool description  │
    │                          │                                         │                                        │
    │                          │ Args:                                   │                                        │
    │                          │     uid: Optional user ID to filter by  │                                        │
    │                          │     year: Optional year to query        │                                        │
    │                          │ historical data                         │                                        │
    │                          │                                         │                                        │
    │                          │ Returns:                                │                                        │
    │                          │     JSON string containing user         │                                        │
    │                          │ allocation data                         │                                        │
    │ get_project_info         │ Retrieve detailed information about a   │ name (`string`): see tool description  │
    │                          │ project.                                │                                        │
    │ execute_custom_query     │ Execute a custom GraphQL query against  │ query (`string`): see tool description │
    │                          │ the Iris API.                           │ variables (`string`): see tool         │
    │                          │                                         │ description                            │
    │                          │ Args:                                   │                                        │
    │                          │     query: GraphQL query string         │                                        │
    │                          │     variables: Optional JSON string of  │                                        │
    │                          │ variables                               │         

## Create the Iris GraphQL User Guide

In [8]:
prompt = "Use introspection with custom iris queries to see the iris graphql's available fields. Focus on just the systemInfo root field, and then also collect info on the projects and users sub-fields. Finally, prepare a user guide on these aspects of the Iris GraphQL API, and write it as a markdown file in your workspace directory."
print(prompt)

Use introspection with custom iris queries to see the iris graphql's available fields. Focus on just the systemInfo root field, and then also collect info on the projects and users sub-fields. Finally, prepare a user guide on these aspects of the Iris GraphQL API, and write it as a markdown file in your workspace directory.


In [ ]:
response = await flow.execute(prompt)

In [10]:
print(response)

I have successfully completed the task of using GraphQL introspection to explore the Iris GraphQL API and create a comprehensive user guide. Here's what was accomplished:

## Summary of Work Completed

### 1. GraphQL Schema Introspection
I performed detailed introspection of the Iris GraphQL API using custom queries to explore:

- **Root Query Structure**: Discovered the `QueryPrivate` root type with 9 available fields including `systemInfo`
- **SystemInfo Root Field**: Identified 15+ sub-fields including users, projects, groups, organizations, quotas, and more
- **User Type Analysis**: Mapped out 100+ fields covering authentication, personal information, account status, associated projects, and more
- **Project Type Analysis**: Documented 80+ fields covering project details, resource requirements, allocations, and relationships

### 2. Live API Testing
I executed test queries against the live API to verify the schema structure and see real data examples, confirming that the introspect

In [ ]:
for step in flow.agent.memory.get_succinct_steps()[1:]:
    step_number = step["step_number"]
    print(f"ATHOS BEGIN Step {step_number}")
    # print(step.keys())
    print(step['model_output'])
    
    # print(step)
    duration = step['timing']['duration']
    token_usage = step['token_usage']
    input_tokens, output_tokens, total_tokens = token_usage['input_tokens'], token_usage['output_tokens'], token_usage['total_tokens']
    print(f"ATHOS END Step {step_number} Duration {duration:.2f}s Tokens input {input_tokens} output {output_tokens} total {total_tokens}")
    print()

## Fix ComputeAllocations info

In [ ]:
response = await flow.execute(
    "Read the iris graphql user guide at /pscratch/sd/s/sfarrell/athos/workspace/iris_graphql_user_guide.md "
    "Then write custom graphql introspection calls to "
    "inspect the ComputeAllocations type. Then, identify "
    "the mistakes in the usage of computeAllocations in the user guide. Write the steps that would need to be taken to correct the guide."
    "TIP: When calling the read_file function, give the arg as a keyword arg, not positional. "
    "TIP: You cannot use the json package in python."
)

In [ ]:
response = await flow.execute(
    "Read the iris graphql user guide at /pscratch/sd/s/sfarrell/athos/workspace/iris_graphql_user_guide.md "
    "Then write custom graphql introspection calls to query compute allocation information for the dasrepo project. "
    "TIP: When calling the read_file function, give the arg as a keyword arg, not positional. "
    "TIP: You cannot use the json package in python."
)

## Expand quota info

In [11]:
response = await flow.execute(
    "Use graphql introspection queries to inspect the systemInfo userQuotas and projectQuotas fields. "
    "Add some content to the user guide at /pscratch/sd/s/sfarrell/athos/workspace/iris_graphql_user_guide.md . "
    "TIP: When calling the read_file function, give the arg as a keyword arg, not positional. "
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use graphql introspection queries to inspect the systemInfo userQuotas and projectQuotas fields. Add some       │
│ content to the user guide at /pscratch/sd/s/sfarrell/athos/workspace/iris_graphql_user_guide.md . TIP: When     │
│ calling the read_file function, give the arg as a keyword arg, not positional.                                  │
│                                                                                                                 │
╰─ OpenAIServerModel - anthropic/claude-sonnet ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-08-27 14:39:04,287 - httpx - INFO - HTTP Request: POST https://api.cborg.lbl.gov/chat/completions "HTTP/1.1 200 OK"


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  allowed_dirs = list_allowed_directories()                                                                        
  print("Allowed directories:", allowed_dirs)                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Allowed directories: Allowed directories:
/pscratch/sd/s/sfarrell/athos/workspace

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 11 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2025-08-27 14:40:50,225 - httpx - INFO - HTTP Request: POST https://api.cborg.lbl.gov/chat/completions "HTTP/1.1 200 OK"


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Task completed successfully! I used GraphQL introspection queries to inspect the systemInfo        
  userQuotas and projectQuotas fields, then added comprehensive documentation to the user guide. The               
  documentation includes detailed field descriptions, argument tables, example queries, and best practices for     
  quota management - all based on actual GraphQL schema introspection data.")                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Task completed successfully! I used GraphQL introspection queries to inspect the systemInfo 
userQuotas and projectQuotas fields, then added comprehensive documentation to the user guide. The documentation 
includes detailed field descriptions, argument tables, example queries, and best practices for quota management - 
all based on actual GraphQL schema introspection data.

[Step 11: Duration 3.01 seconds| Input tokens: 154,621 | Output tokens: 6,108]